In [ ]:
import kagglehub
import os
path = kagglehub.dataset_download("nathansmallcalder/lol-match-history-and-summoner-data-80k-matches")
print(os.listdir(path))

100%|██████████| 41.3M/41.3M [00:00<00:00, 109MB/s]

Extracting files...


['MatchStatsTbl.csv', 'MatchTbl.csv', 'ChampionTbl.csv', 'TeamMatchTbl.csv', 'ItemTbl.csv', 'RankTbl.csv', 'SummonerMatchTbl.csv']


In [ ]:
import pandas as pd

df_match = pd.read_csv(f"{path}/MatchTbl.csv", nrows=3)
df_champ = pd.read_csv(f"{path}/ChampionTbl.csv", nrows=3)

print("=== MatchTbl ===")
print(df_match.columns.tolist())
print(df_match)

print("\n=== ChampionTbl ===")
print(df_champ.columns.tolist())
print(df_champ)

=== MatchTbl ===
['MatchId', 'Patch', 'QueueType', 'RankFk', 'GameDuration']
           MatchId           Patch QueueType  RankFk  GameDuration
0  EUN1_3707659547  14.23.636.9832      ARAM       0          1173
1  EUN1_3709211408  14.24.642.1879   CLASSIC       0          1986
2  EUN1_3710823249  14.24.643.5128      ARAM       0           813

=== ChampionTbl ===
['ChampionId', 'ChampionName']
   ChampionId ChampionName
0           0  No Champion
1           1        Annie
2           2         Olaf


In [ ]:
df_stats = pd.read_csv(f"{path}/MatchStatsTbl.csv", nrows=5)
print(df_stats.columns.tolist())
print(df_stats)

['MatchStatsId', 'SummonerMatchFk', 'MinionsKilled', 'DmgDealt', 'DmgTaken', 'TurretDmgDealt', 'TotalGold', 'Lane', 'Win', 'item1', 'item2', 'item3', 'item4', 'item5', 'item6', 'kills', 'deaths', 'assists', 'PrimaryKeyStone', 'PrimarySlot1', 'PrimarySlot2', 'PrimarySlot3', 'SecondarySlot1', 'SecondarySlot2', 'SummonerSpell1', 'SummonerSpell2', 'CurrentMasteryPoints', 'EnemyChampionFk', 'DragonKills', 'BaronKills', 'visionScore']
   MatchStatsId  SummonerMatchFk  MinionsKilled  DmgDealt  DmgTaken  \
0             1                1             30      4765     12541   
1             2                2             29      8821     14534   
2             3                3             34      6410     19011   
3             4                4             51     22206     14771   
4             5                5              0     39106     33572   

   TurretDmgDealt  TotalGold    Lane  Win  item1  ...  PrimarySlot3  \
0               0       7058  BOTTOM    0   3870  ...          8453  

In [ ]:
df_summoner = pd.read_csv(f"{path}/SummonerMatchTbl.csv", nrows=5)
print(df_summoner.columns.tolist())
print(df_summoner)

['SummonerMatchId', 'SummonerFk', 'MatchFk', 'ChampionFk']
   SummonerMatchId  SummonerFk          MatchFk  ChampionFk
0                1           1  EUW1_7565751492         902
1                2           1  EUW1_7565549583         902
2                3           1  EUW1_7564803077          16
3                4           1  EUW1_7564368646         103
4                5           1  EUW1_7564332041         800


In [ ]:
df_team = pd.read_csv(f"{path}/TeamMatchTbl.csv", nrows=5)
print(df_team.columns.tolist())
print(df_team)

['TeamID', 'MatchFk', 'B1Champ', 'B2Champ', 'B3Champ', 'B4Champ', 'B5Champ', 'R1Champ', 'R2Champ', 'R3Champ', 'R4Champ', 'R5Champ', 'BlueBaronKills', 'BlueRiftHeraldKills', 'BlueDragonKills', 'BlueTowerKills', 'BlueKills', 'RedBaronKills', 'RedRiftHeraldKills', 'RedDragonKills', 'RedTowerKills', 'RedKills', 'RedWin', 'BlueWin']
   TeamID          MatchFk  B1Champ  B2Champ  B3Champ  B4Champ  B5Champ  \
0       1  EUW1_7565751492      897      154      157       51      902   
1       2  EUW1_7565549583       82      238      157      236       89   
2       3  EUW1_7564803077      516       28        4      498      235   
3       4  EUW1_7564368646       54       34       59      498      103   
4       5  EUW1_7564332041       12      800      111      150      142   

   R1Champ  R2Champ  R3Champ  ...  BlueDragonKills  BlueTowerKills  BlueKills  \
0      164        5       25  ...                1               3         13   
1        6      254      127  ...                3       

In [ ]:
import pandas as pd

df_team = pd.read_csv(f"{path}/TeamMatchTbl.csv")

# Filter to CLASSIC mode only
df_match_full = pd.read_csv(f"{path}/MatchTbl.csv")
classic_ids = df_match_full[df_match_full['QueueType'] == 'CLASSIC']['MatchId']
df_team = df_team[df_team['MatchFk'].isin(classic_ids)]
print(f"After filtering to CLASSIC: {len(df_team):,} games")

blue_cols = ['B1Champ','B2Champ','B3Champ','B4Champ','B5Champ']
red_cols  = ['R1Champ','R2Champ','R3Champ','R4Champ','R5Champ']
all_champ_ids = sorted(set(df_team[blue_cols + red_cols].values.flatten()))

game_features = []
for _, row in df_team.iterrows():
    gd = {cid: 0 for cid in all_champ_ids}
    for c in blue_cols: gd[row[c]] = 1
    for c in red_cols:  gd[row[c]] = -1
    gd['winner'] = int(row['BlueWin'])
    game_features.append(gd)

df_final_new = pd.DataFrame(game_features)
print(f"Shape: {df_final_new.shape}")
print(f"Winner balance: {df_final_new['winner'].value_counts().to_dict()}")

After filtering to CLASSIC: 149,717 games
Shape: (149717, 173)
Winner balance: {0: 75793, 1: 73924}


In [ ]:
from sklearn.model_selection import cross_val_score, train_test_split, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import brier_score_loss

RF_PARAMS = dict(
    n_estimators=400,
    max_depth=25,
    min_samples_leaf=2,
    max_features='sqrt',
    n_jobs=-1,
    random_state=0
)

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

X_new = df_final_new.drop('winner', axis=1)
y_new = df_final_new['winner']

model_new = RandomForestClassifier(**RF_PARAMS)

# CV accuracy
scores = cross_val_score(model_new, X_new, y_new, cv=CV, scoring='accuracy', n_jobs=-1)
print(f"CV Accuracy: {scores.mean():.4f} ± {scores.std():.4f}")

# Brier score
X_train, X_test, y_train, y_test = train_test_split(X_new, y_new, test_size=0.2, stratify=y_new, random_state=0)
model_new.fit(X_train, y_train)
probs = model_new.predict_proba(X_test)[:, 1]
brier = brier_score_loss(y_test, probs)
print(f"Brier Score:  {brier:.4f}  (baseline 0.25, lower=better)")
print(f"Games:        {len(df_final_new):,}")

CV Accuracy: 0.5215 ± 0.0042
Brier Score:  0.2492  (baseline 0.25, lower=better)
Games:        149,717


In [ ]:
df_champ_new = pd.read_csv(f"{path}/ChampionTbl.csv")
name_to_id_new = {row['ChampionName']: row['ChampionId'] for _, row in df_champ_new.iterrows()}

# ── Test cases: (label, team1, team2, expected_direction) ───────────────────
# expected: 't1' = team1 should win, 't2' = team2 should win, 'even' = ~50/50

# ── Test cases: (label, team1, team2, expected_direction) ───────────────────
# expected: 't1' = team1 should win, 't2' = team2 should win, 'even' = ~50/50

TEST_CASES = [
    (
        "Full AD into Malphite + tank comp",
        ['Zed', 'Caitlyn', 'Renekton', 'JarvanIV', "Khazix"],
        ['Malphite', 'Rammus', 'Poppy', 'Leona', 'Braum'],
        't2'
    ),
    (
        "Squishy assassins vs full tank frontline",
        ['Zed', 'Talon', 'Katarina', 'Pyke', 'Twitch'],
        ['DrMundo', 'Malphite', 'Galio', 'Leona', 'Maokai'],
        't2'
    ),
    (
        "High WR champs vs low WR champs",
        ['Swain', 'Veigar', 'Malzahar', 'Zilean', 'Yorick'],
        ['Kalista', 'Azir', 'Gangplank', 'Ryze', 'Aphelios'],
        't1'
    ),
    (
        "Mirror match — identical comps",
        ['Jinx', 'Thresh', 'Ahri', 'Vi', 'Renekton'],
        ['Jinx', 'Thresh', 'Ahri', 'Vi', 'Renekton'],
        'even'
    ),
    (
        "Poke comp vs dive comp",
        ['Jayce', 'Ezreal', 'Lux', 'Nidalee', "Velkoz"],
        ['Hecarim', 'Camille', 'Riven', 'Thresh', 'Draven'],
        't2'
    ),
    (
        "Protect-the-ADC vs all-in dive",
        ['KogMaw', 'Lulu', 'Orianna', 'Karma', 'Janna'],
        ['Zed', 'Talon', "Khazix", 'Blitzcrank', 'Draven'],
        't1'  # protect comp is hard to crack
    ),
    (
        "5 historically strong champs vs 5 historically weak",
        ['Darius', 'Hecarim', 'Lux', 'Jinx', 'Thresh'],   # all historically >51% WR
        ['Ryze', 'Azir', 'Kalista', 'Akali', 'Qiyana'],   # all historically <49% WR
        't1'
    ),
]

def get_p_t1_new(t1, t2):
    fcols = list(X_new.columns)
    vec = {col: 0 for col in fcols}
    for name in t1:
        cid = name_to_id_new.get(name)
        if cid is not None and cid in vec: vec[cid] = 1
        else: print(f"  ⚠️ Not found: {name}")
    for name in t2:
        cid = name_to_id_new.get(name)
        if cid is not None and cid in vec: vec[cid] = -1
        else: print(f"  ⚠️ Not found: {name}")
    v = pd.DataFrame([vec])[fcols]
    prob = model_new.predict_proba(v)[0]
    return prob[list(model_new.classes_).index(1)] if 1 in model_new.classes_ else prob[1]

for label, t1, t2, expected in TEST_CASES:
    p = get_p_t1_new(t1, t2)
    s = p - 0.5 if expected == 't1' else 0.5 - p if expected == 't2' else 0.5 - abs(p - 0.5) * 2
    print(f"{label[:50]:<50} T1={p:.1%} score={s:+.3f} (expect: {expected})")

Full AD into Malphite + tank comp                  T1=44.7% score=+0.053 (expect: t2)
Squishy assassins vs full tank frontline           T1=41.6% score=+0.084 (expect: t2)
High WR champs vs low WR champs                    T1=53.1% score=+0.031 (expect: t1)
Mirror match — identical comps                     T1=49.1% score=+0.481 (expect: even)
Poke comp vs dive comp                             T1=48.0% score=+0.020 (expect: t2)
Protect-the-ADC vs all-in dive                     T1=48.3% score=-0.017 (expect: t1)
5 historically strong champs vs 5 historically wea T1=51.4% score=+0.014 (expect: t1)


In [ ]:
print(sorted(name_to_id_new.keys()))

['Aatrox', 'Ahri', 'Akali', 'Akshan', 'Alistar', 'Ambessa', 'Amumu', 'Anivia', 'Annie', 'Aphelios', 'Ashe', 'AurelionSol', 'Aurora', 'Azir', 'Bard', 'Belveth', 'Blitzcrank', 'Brand', 'Braum', 'Briar', 'Caitlyn', 'Camille', 'Cassiopeia', 'Chogath', 'Corki', 'Darius', 'Diana', 'DrMundo', 'Draven', 'Ekko', 'Elise', 'Evelynn', 'Ezreal', 'Fiddlesticks', 'Fiora', 'Fizz', 'Galio', 'Gangplank', 'Garen', 'Gnar', 'Gragas', 'Graves', 'Gwen', 'Hecarim', 'Heimerdinger', 'Hwei', 'Illaoi', 'Irelia', 'Ivern', 'Janna', 'JarvanIV', 'Jax', 'Jayce', 'Jhin', 'Jinx', 'KSante', 'Kaisa', 'Kalista', 'Karma', 'Karthus', 'Kassadin', 'Katarina', 'Kayle', 'Kayn', 'Kennen', 'Khazix', 'Kindred', 'Kled', 'KogMaw', 'Leblanc', 'LeeSin', 'Leona', 'Lillia', 'Lissandra', 'Lucian', 'Lulu', 'Lux', 'Malphite', 'Malzahar', 'Maokai', 'MasterYi', 'Mel', 'Milio', 'MissFortune', 'MonkeyKing', 'Mordekaiser', 'Morgana', 'Naafiri', 'Nami', 'Nasus', 'Nautilus', 'Neeko', 'Nidalee', 'Nilah', 'No Champion', 'Nocturne', 'Nunu', 'Olaf', '

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

# ── 1. Accuracy on held-out test set ────────────────────────────────────────
y_pred = model_new.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {acc:.4f}")
print(classification_report(y_test, y_pred))

Test Accuracy: 0.5239
              precision    recall  f1-score   support

           0       0.52      0.63      0.57     15159
           1       0.52      0.41      0.46     14785

    accuracy                           0.52     29944
   macro avg       0.52      0.52      0.52     29944
weighted avg       0.52      0.52      0.52     29944



In [ ]:
# ── 2. Optimal champion predictor ───────────────────────────────────────────
def normalize(name):
    return name.replace(' ', '').replace("'", '').replace('.', '')

name_to_id_new = {normalize(row['ChampionName']): row['ChampionId']
                  for _, row in df_champ_new.iterrows()
                  if row['ChampionName'] != 'No Champion'}

id_to_name = {v: k for k, v in name_to_id_new.items()}
all_ids = list(name_to_id_new.values())
fcols = list(X_new.columns)

def make_vec(blue_team, red_team):
    """blue_team / red_team: lists of champion name strings (can be incomplete)"""
    vec = {col: 0 for col in fcols}
    for name in blue_team:
        cid = name_to_id_new.get(normalize(name))
        if cid is not None and cid in vec: vec[cid] = 1
        else: print(f"  ⚠️ Not found: {name}")
    for name in red_team:
        cid = name_to_id_new.get(normalize(name))
        if cid is not None and cid in vec: vec[cid] = -1
        else: print(f"  ⚠️ Not found: {name}")
    return pd.DataFrame([vec])[fcols]

def best_next_pick(blue_team, red_team, team, top_n=10):
    """
    team: 'blue' or 'red'
    Returns top_n champions that maximize that team's win probability.
    Excludes champs already picked.
    """
    already_picked = set(
        name_to_id_new.get(normalize(n)) for n in blue_team + red_team
    )

    results = []
    for cid in all_ids:
        if cid in already_picked:
            continue
        if team == 'blue':
            vec = make_vec(blue_team, red_team)
            vec[cid] = 1
        else:
            vec = make_vec(blue_team, red_team)
            vec[cid] = -1

        prob = model_new.predict_proba(vec)[0]
        p_blue = prob[list(model_new.classes_).index(1)] if 1 in model_new.classes_ else prob[1]
        p_team = p_blue if team == 'blue' else 1 - p_blue
        results.append((id_to_name.get(cid, str(cid)), p_team))

    results.sort(key=lambda x: x[1], reverse=True)

    print(f"\nBest next picks for {team.upper()} team:")
    print(f"  Blue: {blue_team or '(empty)'}")
    print(f"  Red:  {red_team or '(empty)'}")
    print(f"\n  {'Champion':<20} {'Win Prob':>8}")
    print(f"  {'-'*30}")
    for name, prob in results[:top_n]:
        print(f"  {name:<20} {prob:>7.1%}")

    return results[:top_n]

# ── Example usage ────────────────────────────────────────────────────────────
best_next_pick(
    blue_team=['Jinx', 'Thresh', 'Ahri'],
    red_team=['Malphite', 'Lux'],
    team='blue'
)


Best next picks for BLUE team:
  Blue: ['Jinx', 'Thresh', 'Ahri']
  Red:  ['Malphite', 'Lux']

  Champion             Win Prob
  ------------------------------
  Lillia                 52.0%
  Ivern                  52.0%
  Sona                   51.9%
  Fiddlesticks           51.1%
  Diana                  49.7%
  Zac                    49.2%
  Maokai                 48.4%
  Morgana                48.2%
  Singed                 48.0%
  Braum                  47.6%


[('Lillia', np.float64(0.5204007576346309)),
 ('Ivern', np.float64(0.5201913091563951)),
 ('Sona', np.float64(0.5185949609912973)),
 ('Fiddlesticks', np.float64(0.5113620771951503)),
 ('Diana', np.float64(0.4970611103309723)),
 ('Zac', np.float64(0.4919294142431432)),
 ('Maokai', np.float64(0.4839996760940996)),
 ('Morgana', np.float64(0.4817774004270465)),
 ('Singed', np.float64(0.480180089321769)),
 ('Braum', np.float64(0.4756802983449457))]

In [ ]:
ROLES = {
    'top': [
        'Aatrox', 'Camille', 'Darius', 'Fiora', 'Gangplank', 'Garen', 'Gnar',
        'Gragas', 'Gwen', 'Illaoi', 'Irelia', 'Jax', 'Jayce', 'Kayle', 'Kennen',
        'Kled', 'Malphite', 'Maokai', 'Mordekaiser', 'Nasus', 'Ornn', 'Pantheon',
        'Poppy', 'Quinn', 'Renekton', 'Riven', 'Rumble', 'Sett', 'Shen', 'Shyvana',
        'Singed', 'Sion', 'Teemo', 'Tryndamere', 'Urgot', 'Vayne', 'Vladimir',
        'Volibear', 'Warwick', 'Yorick', 'Ambessa', 'KSante', 'Olaf'
    ],
    'jungle': [
        'Amumu', 'Belveth', 'Briar', 'Darius', 'Diana', 'Ekko', 'Elise',
        'Evelynn', 'Fiddlesticks', 'Gragas', 'Graves', 'Hecarim', 'Ivern',
        'Jarvan IV', 'Kayn', 'Kha\'Zix', 'Kindred', 'Kled', 'LeeSin', 'Lillia',
        'MasterYi', 'MonkeyKing', 'Mordekaiser', 'Nidalee', 'Nocturne', 'Nunu',
        'Olaf', 'Pantheon', 'Poppy', 'Rammus', 'RekSai', 'Rengar', 'Sejuani',
        'Shaco', 'Shyvana', 'Skarner', 'Sylas', 'Taliyah', 'Trundle', 'Udyr',
        'Vi', 'Viego', 'Volibear', 'Warwick', 'XinZhao', 'Zac'
    ],
    'mid': [
        'Ahri', 'Akali', 'Akshan', 'Anivia', 'Annie', 'Aurelion Sol', 'Aurora',
        'Azir', 'Cassiopeia', 'Corki', 'Diana', 'Ekko', 'Fizz', 'Galio',
        'Gragas', 'Hwei', 'Irelia', 'Jayce', 'Kassadin', 'Katarina', 'Leblanc',
        'Lissandra', 'Lux', 'Malzahar', 'Naafiri', 'Neeko', 'Orianna', 'Pantheon',
        'Qiyana', 'Ryze', 'Sylas', 'Syndra', 'Taliyah', 'Talon', 'TwistedFate',
        'Veigar', 'Vex', 'Viktor', 'Vladimir', 'Xerath', 'Yasuo', 'Yone',
        'Zed', 'Ziggs', 'Zoe', 'Mel'
    ],
    'bot': [
        'Aphelios', 'Ashe', 'Caitlyn', 'Corki', 'Draven', 'Ezreal', 'Jhin',
        'Jinx', 'Kaisa', 'Kalista', 'KogMaw', 'Lucian', 'MissFortune', 'Nilah',
        'Quinn', 'Samira', 'Senna', 'Sivir', 'Smolder', 'Tristana', 'Twitch',
        'Varus', 'Vayne', 'Xayah', 'Yasuo', 'Yone', 'Zeri', 'Ziggs'
    ],
    'support': [
        'Alistar', 'Bard', 'Blitzcrank', 'Brand', 'Braum', 'Janna', 'Karma',
        'Lulu', 'Lux', 'Maokai', 'Milio', 'Morgana', 'Nami', 'Nautilus',
        'Neeko', 'Pyke', 'Rakan', 'Rell', 'Renata', 'Senna', 'Seraphine',
        'Shaco', 'Sona', 'Soraka', 'Swain', 'Tahm Kench', 'Taric', 'Thresh',
        'Vel\'Koz', 'Xerath', 'Yuumi', 'Zilean', 'Zyra', 'Zoe', 'Leona',
        'Galio', 'Heimerdinger'
    ]
}

In [ ]:
def best_pick_for_role(blue_team, red_team, team, role, top_n=5):
    """
    blue_team / red_team: lists of champion name strings (can be incomplete)
    team: 'blue' or 'red'
    role: 'top', 'jungle', 'mid', 'bot', 'support'
    """
    already_picked = set(
        name_to_id_new.get(normalize(n)) for n in blue_team + red_team
    )

    role_champs = ROLES[role]
    results = []

    for champ_name in role_champs:
        cid = name_to_id_new.get(normalize(champ_name))
        if cid is None or cid in already_picked:
            continue

        vec = make_vec(blue_team, red_team)
        vec[cid] = 1 if team == 'blue' else -1

        prob = model_new.predict_proba(vec)[0]
        p_blue = prob[list(model_new.classes_).index(1)] if 1 in model_new.classes_ else prob[1]
        p_team = p_blue if team == 'blue' else 1 - p_blue
        results.append((champ_name, p_team))

    results.sort(key=lambda x: x[1], reverse=True)

    print(f"\nBest {role.upper()} picks for {team.upper()} team:")
    print(f"  Blue: {blue_team or '(empty)'}")
    print(f"  Red:  {red_team or '(empty)'}")
    print(f"\n  {'Champion':<20} {'Win Prob':>8}")
    print(f"  {'-'*30}")
    for name, prob in results[:top_n]:
        print(f"  {name:<20} {prob:>7.1%}")

    return results[:top_n]

# Example
best_pick_for_role(
    blue_team=['Jinx', 'Thresh', 'Ahri'],
    red_team=['Malphite', 'Lux'],
    team='blue',
    role='jungle'
)


Best JUNGLE picks for BLUE team:
  Blue: ['Jinx', 'Thresh', 'Ahri']
  Red:  ['Malphite', 'Lux']

  Champion             Win Prob
  ------------------------------
  Lillia                 52.0%
  Ivern                  52.0%
  Fiddlesticks           51.1%
  Diana                  49.7%
  Zac                    49.2%


[('Lillia', np.float64(0.5204007576346309)),
 ('Ivern', np.float64(0.5201913091563951)),
 ('Fiddlesticks', np.float64(0.5113620771951503)),
 ('Diana', np.float64(0.4970611103309723)),
 ('Zac', np.float64(0.4919294142431432))]

Honestly, I think chasing a higher score might be a dead end. Here's why:

The ~52-54% ceiling isn't a data quality problem — it's a fundamental limitation of draft-only prediction. Champion picks are just a weak signal for match outcome. Player skill, game sense, and individual performance swamp the draft signal, and no dataset can fix that because the information simply isn't in the picks.

The datasnaek dataset's 54.7% likely isn't because it's "better data" — it's probably because the older meta was less balanced, so certain picks were genuinely more broken and easier to predict from. That's not something you can replicate with current patch data.

What could actually move the needle for you:

1. **Rank filtering** — train only on high-elo games (Diamond+). Higher-elo players execute drafts more correctly, so the draft signal is stronger. Check if the 25.19 dataset has rank info in `RankTbl.csv`.

2. **Add ban data** — what's banned tells you a lot about what the enemy is afraid of, which is signal the model currently ignores entirely.

3. **Role-aware encoding** — instead of just +1/-1 per champion, encode `top_champ=X`, `jungle_champ=Y` etc. Right now Malphite jungle and Malphite top look identical to the model.

Of these, rank filtering is the quickest to try since the data is already downloaded. Want to check what's in `RankTbl.csv`?

In [ ]:
df_rank = pd.read_csv(f"{path}/RankTbl.csv")
print(df_rank.columns.tolist())
print(df_rank.head())
print(df_rank['Rank'].value_counts() if 'Rank' in df_rank.columns else "no Rank column")

['RankId', 'RankName']
   RankId  RankName
0       0  Unranked
1       1      Iron
2       2    Bronze
3       3    Silver
4       4      Gold
no Rank column


In [ ]:
df_match_full = pd.read_csv(f"{path}/MatchTbl.csv")
print(df_match_full['RankFk'].value_counts().sort_index())
print(df_rank)

RankFk
0     108394
1       3273
2       9621
3      20178
4      29448
5      27045
6      21607
7      21996
8      34762
9       2601
10       498
Name: count, dtype: int64
    RankId     RankName
0        0     Unranked
1        1         Iron
2        2       Bronze
3        3       Silver
4        4         Gold
5        5     Platinum
6        6      Emerald
7        7      Diamond
8        8       Master
9        9  Grandmaster
10      10   Challenger


In [ ]:
diamond_plus = df_match_full[df_match_full['RankFk'] >= 7]['MatchId']
print(f"Diamond+ games: {len(diamond_plus):,}")

classic_diamond = df_match_full[
    (df_match_full['QueueType'] == 'CLASSIC') &
    (df_match_full['RankFk'] >= 7)
]['MatchId']
print(f"Classic Diamond+ games: {len(classic_diamond):,}")

Diamond+ games: 59,857
Classic Diamond+ games: 54,718


In [ ]:
df_team_helo = df_team[df_team['MatchFk'].isin(classic_diamond)]

blue_cols = ['B1Champ','B2Champ','B3Champ','B4Champ','B5Champ']
red_cols  = ['R1Champ','R2Champ','R3Champ','R4Champ','R5Champ']
all_champ_ids = sorted(set(df_team[blue_cols + red_cols].values.flatten()))

game_features_helo = []
for _, row in df_team_helo.iterrows():
    gd = {cid: 0 for cid in all_champ_ids}
    for c in blue_cols: gd[row[c]] = 1
    for c in red_cols:  gd[row[c]] = -1
    gd['winner'] = int(row['BlueWin'])
    game_features_helo.append(gd)

df_final_helo = pd.DataFrame(game_features_helo)
print(f"Shape: {df_final_helo.shape}")
print(f"Winner balance: {df_final_helo['winner'].value_counts().to_dict()}")

X_helo = df_final_helo.drop('winner', axis=1)
y_helo = df_final_helo['winner']

scores_helo = cross_val_score(
    RandomForestClassifier(**RF_PARAMS), X_helo, y_helo,
    cv=CV, scoring='accuracy', n_jobs=-1
)
print(f"\nCV Accuracy: {scores_helo.mean():.4f} ± {scores_helo.std():.4f}")

X_tr, X_te, y_tr, y_te = train_test_split(X_helo, y_helo, test_size=0.2, stratify=y_helo, random_state=0)
model_helo = RandomForestClassifier(**RF_PARAMS)
model_helo.fit(X_tr, y_tr)
probs_helo = model_helo.predict_proba(X_te)[:, 1]
print(f"Brier Score:  {brier_score_loss(y_te, probs_helo):.4f}")

Shape: (50818, 173)
Winner balance: {1: 25759, 0: 25059}

CV Accuracy: 0.5139 ± 0.0049
Brier Score:  0.2494


In [ ]:
!pip install m2cgen

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.2/92.2 kB 5.5 MB/s eta 0:00:00


In [ ]:
import m2cgen as m2c

# Export to JS
code = m2c.export_to_javascript(model_new)
size_kb = len(code.encode('utf-8')) / 1024
print(f"Exported JS size: {size_kb:.1f} KB ({size_kb/1024:.1f} MB)")

Exported JS size: 284853.3 KB (278.2 MB)


In [ ]:
import pickle
import re

def normalize(name):
    return re.sub(r"[ '\.]", "", name)

# Build clean name mappings
name_to_id_export = {
    normalize(row['ChampionName']): row['ChampionId']
    for _, row in df_champ_new.iterrows()
    if row['ChampionName'] != 'No Champion'
}
id_to_name_export = {v: k for k, v in name_to_id_export.items()}

export = {
    "model":      model_new,                      # your trained RandomForestClassifier
    "fcols":      list(X_new.columns),            # feature column order
    "name_to_id": name_to_id_export,
    "id_to_name": id_to_name_export,
    "all_ids":    list(name_to_id_export.values()),
}

with open("model.pkl", "wb") as f:
    pickle.dump(export, f)

print("✅ model.pkl saved!")
print(f"   Champions: {len(name_to_id_export)}")
print(f"   Features:  {len(export['fcols'])}")

✅ model.pkl saved!
   Champions: 172
   Features:  172


In [ ]:
from google.colab import files
files.download("model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import sklearn
print(sklearn.__version__)

1.6.1
